## Pydantic

- pydantic.BaseModel을 상속받는 클래스로 모델 생성

In [1]:
from pydantic import BaseModel

# 문서 한개를 담을 수 있는 클래스 생성 (타입 체크 원해)
class Document(BaseModel):
    # __init__ 을 생략해도, BaseModel이 아래 필드 선언을 읽고 자동으로 생성.
    doc_id: str
    title: str
    version: str
    security_level: str

doc = Document(
    doc_id="DOC-HR-001",
    title="여비 규정 문서",
    version="1.0",
    security_level="사내공개"
)

print(doc)      # __repr__도 자동으로 정의해준다.
print(doc.title)

doc_id='DOC-HR-001' title='여비 규정 문서' version='1.0' security_level='사내공개'
여비 규정 문서


In [2]:
from pydantic import Field
from datetime import date

# 제약이 붙은 문서 모델
class Document2(BaseModel):
    # Field
    doc_id: str = Field(
        ...,
        pattern=r"DOC-[A-Z]{2,4}-\d{3}$",       # 정규표현식
        description="문서 고유 번호"
    )
    title: str = Field(
        ...,
        min_length = 1,
        max_length = 200,
    )
    version: str = Field(..., pattern=r"^\d+\.\d+$") # 2.0
    security_level: str
    expiry_date: date | None = None                 # 여러 타입
    tags: list[str] = Field(default_factory=list)

doc2 = Document2(
    doc_id = "DOC-HR-002",
    title = "여비 규정 문서",
    version = "1.1",
    security_level = "사내배포",
    tags=["aaa", "bbb"]
)
print(doc2)

doc_id='DOC-HR-002' title='여비 규정 문서' version='1.1' security_level='사내배포' expiry_date=None tags=['aaa', 'bbb']


In [3]:
from enum import Enum
from typing import Literal

# enum은 값 변경 안하는 상수들의 집합
class SecurityLevel(str, Enum):
    # str과 함께 Enum을 상속받아 구현하며, 문자열처럼 사용할 수 있어 JSON변환이 편리하다.
    PUBLIC = "공개"
    INTERNAL = "사내공개"
    CONFIDENTIAL = "대외비"
    SECRET = "기밀"

class Document3(BaseModel):
    doc_id: str
    security_level: SecurityLevel    # Enum 타입으로 지정
    sec_level: Literal["공개", "사내공개", "대외비", "기밀"]

doc3 = Document3(doc_id="DOC-1", security_level=SecurityLevel.CONFIDENTIAL, sec_level="대외비")
print(doc3)
print(doc3.security_level.value)

doc_id='DOC-1' security_level=<SecurityLevel.CONFIDENTIAL: '대외비'> sec_level='대외비'
대외비


In [4]:
class Document4(BaseModel):
    doc_id: str
    title: str
    effective_date: date
    tags: list[str] = Field(default_factory=list)

doc4 = Document4(
    doc_id="DOC-HR-003",
    title="국내 출장 여비 규정",
    effective_date=date(2026, 9, 7),
    tags=["인사", "출장", "여비"]
)

print(doc4)

# 모델(파이썬 객체) -> 딕셔너리
d = doc.model_dump()
print(d)
# 모델 -> JSON
j = doc.model_dump_json(indent=2)   # indent: 들여쓰기 옵션
print(j)


doc_id='DOC-HR-003' title='국내 출장 여비 규정' effective_date=datetime.date(2026, 9, 7) tags=['인사', '출장', '여비']
{'doc_id': 'DOC-HR-001', 'title': '여비 규정 문서', 'version': '1.0', 'security_level': '사내공개'}
{
  "doc_id": "DOC-HR-001",
  "title": "여비 규정 문서",
  "version": "1.0",
  "security_level": "사내공개"
}


In [5]:
# 넘어온 딕셔너리/JSON -> 모델

data = {
    "doc_id": "DOC-SEC-002",
    "title": "정보 보안 지침",
    "effective_date": "2026-09-07",
    "tags": ["보안"]
}

doc_data = Document4.model_validate(data)
print(doc_data)
print(type(doc_data))

# JSON 데이터
data_json = '{"doc_id": "DOC-SEC-002","title" : "정보 보안 지침","effective_date": "2026-09-07", "tags": ["보안"]}'
doc_json = Document4.model_validate_json(data_json)
print(doc_json)

doc_id='DOC-SEC-002' title='정보 보안 지침' effective_date=datetime.date(2026, 9, 7) tags=['보안']
<class '__main__.Document4'>
doc_id='DOC-SEC-002' title='정보 보안 지침' effective_date=datetime.date(2026, 9, 7) tags=['보안']


In [6]:
from pydantic import BaseModel, Field, field_validator
from datetime import date, datetime
from enum import Enum

# 보안 등급: 여러파일에서 사용하므로 enum으로 고정
class SecurityLevel(str, Enum):
    PUBLIC = "공개"
    INTERNAL = "사내공개"
    ONFIDENTIAL = "대외비"
    SECRET = "기밀"

# 요청 모델 : 문서 등록 요청 -> 클라이언트(사용자)가 보내는 것
class DocumentCreate(BaseModel):
    doc_id: str = Field(..., pattern=r"DOC-[A-Z]{2,4}-\d{3}", description="문서 고유 번호(예:DOC-HR-001)")
    title: str = Field(..., min_length=1, max_length=200)
    version: str = Field(..., pattern=r"^\d+\.\d+$", description="예: 1.0")
    department: str = Field(..., min_length=1, max_length=50)
    security_level: SecurityLevel
    effective_date: date
    expiry_date: date | None = None

    # doc_id가 소문자로 들어와도 대문자로 저장하는 처리
    @field_validator("doc_id", mode="before")
    @classmethod
    def upper_doc_id(cls, v):
        return v.upper() if isinstance(v, str) else v

    # title, department의 앞 뒤 공백을 제거
    @field_validator("title", "department")
    @classmethod
    def strip_text(cls, v: str) -> str:
        return v.strip()

# 응답 모델 : 문서 응답 - 서버가 돌려주는 데이터 형태 (요청에는 없던 값이 추가될 수 있다.)
class DocumentOut(BaseModel):
    id: int
    doc_id: str
    title: str
    version: str
    department: str
    security_level: str
    effective_date: str
    expiry_date: str
    is_latest: bool
    chunk_count: int = Field(ge=0)      # 임베딩된 조각 수
    created_at: datetime